# Implement Async batching & caching

Sequential requests make the system slow. Build a powerful summarization where multiple prompts are sent at once.
To optimize implement async parallel execution & multi layer caching to make responses fast and efficient.

1) Implement async parallel requests to improve throughput
2) Use concurrency to avoid rate limit errors.
3) Add 2 caching layers in memory memo and persistent sqlite.
4) Measure latency, throughput and cache hit rates.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("AZURE_OPENAI_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_VERSION")
default_temp = 0.2
model_name = "gpt4o"



In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint= api_endpoint,
    azure_deployment="gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_version=api_version,
    temperature=default_temp)

In [3]:
import asyncio, time, hashlib, json
import pandas as pd
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks import get_openai_callback
model_name = "gpt-4o-mini"
TEMPERATURE =0.2
MAX_CONCURRENCY = 5

# Enable Persistent Langchain Cache

In [4]:
CACHING_DIR ="cache"

In [5]:
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
set_llm_cache(SQLiteCache(database_path=f"{CACHING_DIR}/lc_cache.sqlite3"))

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant. Keep answers short, correct and well-formatted."),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()

In [8]:
import hashlib
import re

#helper function to trim, collapse multiple whitespaces and lowercase the string for cache freindliness
def normalize(s):
    return re.sub(r'\s+', ' ', s.strip().lower())

#helper function to generate cache key
def generate_cache_key(user_input, model, temperature):
    payload = {
        "user_input": normalize(user_input),
        "model": model,
        "temperature": temperature
    }
    payload=json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


In [9]:
sem = asyncio.Semaphore(MAX_CONCURRENCY)
memo_cache = {}

In [15]:
async def async_invoke(question:str, use_memo:bool = True):
    key = generate_cache_key(question, model_name, TEMPERATURE)
    if use_memo and key in memo_cache:
        print("Cache hit (memo)")
        return {"question": question, "answer": memo_cache[key],"latency":0, "prompt_tokens":0, "completion_tokens":0, "total_tokens":0,"source":"memo_cache"}
    
    async with sem:# only 5 questions will be processed concurrently, rest will wait for their turn
        start = time.perf_counter()
        with get_openai_callback() as cb:
            output = await chain.ainvoke({"question": question})
            latency = time.perf_counter() - start
        if use_memo:
            memo_cache[key] = output
        return {"question": question, "answer": output,"latency":latency, "prompt_tokens":cb.prompt_tokens, "completion_tokens":cb.completion_tokens, "total_tokens":cb.total_tokens,"source":"llm"}

In [16]:
#async runner and batch
async def run_batch(questions, use_memo:bool = True):
    tasks = [asyncio.create_task(async_invoke(q, use_memo)) for q in questions]
    results = await asyncio.gather(*tasks)
    return pd.DataFrame(results)


In [17]:
questions = [
    "What is the capital of France?",
    "Who won the FIFA World Cup in 2018?",
    "What is the largest mammal?",
    "Who is the CEO of Tesla?",
    "What is the boiling point of water?"
    "who invented the light bulb?",
    "Tell me about the history of the internet",
    "Who won the FIFA World Cup in 2018?",
     "What is the largest mammal?",
]


In [18]:
start = time.perf_counter()
df = await run_batch(questions)
print(df)


                                            question  \
0                     What is the capital of France?   
1                Who won the FIFA World Cup in 2018?   
2                        What is the largest mammal?   
3                           Who is the CEO of Tesla?   
4  What is the boiling point of water?who invente...   
5          Tell me about the history of the internet   
6                Who won the FIFA World Cup in 2018?   
7                        What is the largest mammal?   

                                              answer   latency  prompt_tokens  \
0                    The capital of France is Paris.  2.005450             34   
1             France won the FIFA World Cup in 2018.  1.752909             38   
2  The largest mammal is the **blue whale** (*Bal...  2.143207             34   
3                     The CEO of Tesla is Elon Musk.  1.880523             34   
4  - Boiling point of water: 100°C (212°F) at sta...  2.633907             41   
5  The hi

In [19]:
df

,question,answer,latency,prompt_tokens,completion_tokens,total_tokens,source
0,What is the capital of France?,The capital of France is Paris.,2.005450,34,8,42,llm
1,Who won the FIFA World Cup in 2018?,France won the FIFA World Cup in 2018.,1.752909,38,12,50,llm
2,What is the largest mammal?,The largest mammal is the **blue whale** (*Bal...,2.143207,34,21,55,llm
3,Who is the CEO of Tesla?,The CEO of Tesla is Elon Musk.,1.880523,34,9,43,llm
4,What is the boiling point of water?who invente...,- Boiling point of water: 100°C (212°F) at sta...,2.633907,41,44,85,llm
5,Tell me about the history of the internet,The history of the internet began in the 1960s...,2.153400,35,137,172,llm
6,Who won the FIFA World Cup in 2018?,France won the FIFA World Cup in 2018.,0.007417,38,12,50,llm
7,What is the largest mammal?,The largest mammal is the **blue whale** (*Bal...,0.981150,34,21,55,llm


In [20]:
memo_cache

{'e6cd41198806ab7c326a1de935c6753ea9681498ef48786e19a14603cd1520f8': 'France won the FIFA World Cup in 2018.',
 '3a0bad006dbbd078250b3c6f2567572c7b18d0122f1e048afba14ad98fa47fc0': 'The CEO of Tesla is Elon Musk.',
 '3802c5c13825e49f69d54a39b6660708b4bebf106da26d1cec8f8ff6dce9ae33': 'The capital of France is Paris.',
 'c555c104ef5689083a6e904cd976d023d23c15637a937f0f1c1b1e2afc6a523f': 'The largest mammal is the **blue whale** (*Balaenoptera musculus*).',
 '6e9e4f2943efd423c4c2a949e863efb1d29cdcd58f154287475e69a3e063f320': '- Boiling point of water: 100°C (212°F) at standard atmospheric pressure (1 atm).\n- Inventor of the light bulb: Thomas Edison is credited with inventing the practical incandescent light bulb.',
 'f674dda1dfe1eb969556f18fd6f99d0d5a5639083b7cd7f43c457b2d30721c87': 'The history of the internet began in the 1960s with ARPANET, a project funded by the U.S. Department of Defense to connect computers for research. In the 1970s, protocols like TCP/IP were developed, enabling

In [23]:
start = time.perf_counter()
df1 = await run_batch(questions)
df1

Cache hit (memo)
Cache hit (memo)
Cache hit (memo)
Cache hit (memo)
Cache hit (memo)
Cache hit (memo)
Cache hit (memo)
Cache hit (memo)


,question,answer,latency,prompt_tokens,completion_tokens,total_tokens,source
0,What is the capital of France?,The capital of France is Paris.,0,0,0,0,memo_cache
1,Who won the FIFA World Cup in 2018?,France won the FIFA World Cup in 2018.,0,0,0,0,memo_cache
2,What is the largest mammal?,The largest mammal is the **blue whale** (*Bal...,0,0,0,0,memo_cache
3,Who is the CEO of Tesla?,The CEO of Tesla is Elon Musk.,0,0,0,0,memo_cache
4,What is the boiling point of water?who invente...,- Boiling point of water: 100°C (212°F) at sta...,0,0,0,0,memo_cache
5,Tell me about the history of the internet,The history of the internet began in the 1960s...,0,0,0,0,memo_cache
6,Who won the FIFA World Cup in 2018?,France won the FIFA World Cup in 2018.,0,0,0,0,memo_cache
7,What is the largest mammal?,The largest mammal is the **blue whale** (*Bal...,0,0,0,0,memo_cache


In [24]:
start = time.perf_counter()
df2 = await run_batch(questions, use_memo=False)
df2

,question,answer,latency,prompt_tokens,completion_tokens,total_tokens,source
0,What is the capital of France?,The capital of France is Paris.,0.034057,34,8,42,llm
1,Who won the FIFA World Cup in 2018?,France won the FIFA World Cup in 2018.,0.032415,38,12,50,llm
2,What is the largest mammal?,The largest mammal is the **blue whale** (*Bal...,0.032372,34,21,55,llm
3,Who is the CEO of Tesla?,The CEO of Tesla is Elon Musk.,0.032226,34,9,43,llm
4,What is the boiling point of water?who invente...,- Boiling point of water: 100°C (212°F) at sta...,0.033839,41,44,85,llm
5,Tell me about the history of the internet,The history of the internet began in the 1960s...,0.015347,35,137,172,llm
6,Who won the FIFA World Cup in 2018?,France won the FIFA World Cup in 2018.,0.014719,38,12,50,llm
7,What is the largest mammal?,The largest mammal is the **blue whale** (*Bal...,0.015445,34,21,55,llm
